In [ ]:
import tkinter as tk
from tkinter import messagebox, filedialog
import google.generativeai as genai
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

# Configure Gemini API
genai.configure(api_key="")  # Replace with your Gemini API key
model = genai.GenerativeModel("models/gemini-1.5-flash-latest")

# Prompt Builder
def build_prompt(name, age, gender, weight, goal, illness):
    return f"""
    Act as a certified health advisor and dietitian. Based on the following user details, provide:
    1. General health advice (exercise, habits, hydration, sleep, etc.)
    2. A simple, nutritious, and achievable diet plan for 1 day (breakfast, lunch, dinner, snacks).
    3. Ensure suggestions align with the user's goal and any existing illness or condition.

    User Info:
    - Name: {name}
    - Age: {age}
    - Gender: {gender}
    - Weight: {weight} kg
    - Goal: {goal} weight
    - Existing Illness: {illness or 'None'}

    Format clearly with headings.
    """

# Generate advice using Gemini
def generate_advice():
    name = name_entry.get().strip()
    age = age_entry.get().strip()
    gender = gender_var.get()
    weight = weight_entry.get().strip()
    goal = goal_var.get()
    illness = illness_text.get("1.0", tk.END).strip()

    if not name or not age or not weight or gender == "Select" or goal == "Select":
        messagebox.showwarning("Input Error", "Please fill all required fields.")
        return

    prompt = build_prompt(name, age, gender, weight, goal, illness)

    try:
        response = model.generate_content(prompt)
        result_text.delete("1.0", tk.END)
        result_text.insert(tk.END, response.text)
    except Exception as e:
        messagebox.showerror("API Error", str(e))

# Export result to PDF
def export_to_pdf():
    content = result_text.get("1.0", tk.END).strip()
    if not content:
        messagebox.showwarning("Export Error", "No content to export. Please generate advice first.")
        return

    file_path = filedialog.asksaveasfilename(defaultextension=".pdf",
                                             filetypes=[("PDF files", "*.pdf")],
                                             title="Save as PDF")
    if file_path:
        try:
            doc = SimpleDocTemplate(file_path, pagesize=letter,
                                    rightMargin=50, leftMargin=50,
                                    topMargin=50, bottomMargin=50)

            styles = getSampleStyleSheet()
            story = []

            for line in content.split("\n"):
                if line.strip() == "":
                    story.append(Spacer(1, 12)) # Space between sections
                else:
                    para = Paragraph(line.strip(), styles["Normal"])
                    story.append(para)
                    story.append(Spacer(1, 6))  # Space between paragraphs

            doc.build(story)
            messagebox.showinfo("Success", f"PDF saved to:\n{file_path}")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to save PDF:\n{str(e)}")

# UI Setup
root = tk.Tk()
root.title("Health Advisor with Diet Plan")
root.geometry("850x750")

tk.Label(root, text="Health Advisor & Diet Planner", font=("Arial", 16, "bold")).pack(pady=10)

form_frame = tk.Frame(root)
form_frame.pack(pady=5)

# Name
tk.Label(form_frame, text="Name:", font=("Arial", 12)).grid(row=0, column=0, sticky="e", padx=5, pady=5)
name_entry = tk.Entry(form_frame, font=("Arial", 12), width=30)
name_entry.grid(row=0, column=1, pady=5)

# Age
tk.Label(form_frame, text="Age:", font=("Arial", 12)).grid(row=1, column=0, sticky="e", padx=5, pady=5)
age_entry = tk.Entry(form_frame, font=("Arial", 12), width=30)
age_entry.grid(row=1, column=1, pady=5)

# Gender
tk.Label(form_frame, text="Gender:", font=("Arial", 12)).grid(row=2, column=0, sticky="e", padx=5, pady=5)
gender_var = tk.StringVar(value="Select")
gender_menu = tk.OptionMenu(form_frame, gender_var, "Male", "Female", "Other")
gender_menu.config(font=("Arial", 12), width=28)
gender_menu.grid(row=2, column=1, pady=5)

# Weight
tk.Label(form_frame, text="Weight (kg):", font=("Arial", 12)).grid(row=3, column=0, sticky="e", padx=5, pady=5)
weight_entry = tk.Entry(form_frame, font=("Arial", 12), width=30)
weight_entry.grid(row=3, column=1, pady=5)

# Goal
tk.Label(form_frame, text="Health Goal:", font=("Arial", 12)).grid(row=4, column=0, sticky="e", padx=5, pady=5)
goal_var = tk.StringVar(value="Select")
goal_menu = tk.OptionMenu(form_frame, goal_var, "Lose", "Gain", "Maintain")
goal_menu.config(font=("Arial", 12), width=28)
goal_menu.grid(row=4, column=1, pady=5)

# Existing Illness (Text Box)
tk.Label(form_frame, text="Existing Illness (if any):", font=("Arial", 12)).grid(row=5, column=0, sticky="ne", padx=5, pady=5)
illness_text = tk.Text(form_frame, font=("Arial", 11), height=3, width=30, wrap="word")
illness_text.grid(row=5, column=1, pady=5)

# Generate Button
tk.Button(root, text="Get Health Advice & Diet Plan", command=generate_advice,
          font=("Arial", 13), bg="#28a745", fg="white", padx=10, pady=6).pack(pady=15)

# Export PDF Button
tk.Button(root, text="Export as PDF", command=export_to_pdf,
          font=("Arial", 12), bg="#007bff", fg="white", padx=10, pady=5).pack()

# Output Text Box
result_text = tk.Text(root, wrap="word", font=("Courier", 11), height=20, width=95)
result_text.pack(padx=10, pady=10)

root.mainloop()
